## Model Selection Strategy

To build a robust Human Activity Recognition system, models were trained and evaluated across different machine learning paradigms and computing environments.

### Training Workflow

The project was divided into three phases:

1. **Traditional Machine Learning Models**
   - Logistic Regression
   - Support Vector Machine (SVM)
   - K-Nearest Neighbors (KNN)

2. **Bagging-Based Ensemble Models**
   - Random Forest
   - Extra Trees Classifier

3. **Boosting-Based Ensemble Models**
   - XGBoost
   - LightGBM
   - CatBoost

Due to computational limitations on the local machine, different environments were used for training:

- Traditional machine learning models were trained in **Google Colab**(https://colab.research.google.com/drive/1kI5ptGl8JAhbofSTckkJcK-SZUt_i6ga).
- Bagging models were trained on the **local machine**.
- Boosting models were trained in **Kaggle** to utilize faster CPU resources.(https://www.kaggle.com/code/adnansaqib1212/notebook3ab1532f49/edit)

### Model Performance Summary

| Category | Model | F1 Macro Score |
|-----------|---------|-------------:|
| Traditional ML | Logistic Regression | 90.52% |
| Traditional ML | SVM | 93.67% |
| Traditional ML | KNN | 93.13% |
| Bagging | Random Forest | 93.10% |
| Bagging | Extra Trees | **94.20%** |
| Boosting | XGBoost | **93.88%** |
| Boosting | LightGBM | 92.69% |
| Boosting | CatBoost | 90.03% |

### Best Models Selected

Based on performance and diversity, the following models were selected:

- **Extra Trees Classifier** (Best Bagging Model)
- **XGBoost** (Best Boosting Model)
- **Logistic Regression** (Best Linear Model)

These models were chosen because they represent different learning strategies and are expected to complement each other effectively in an ensemble architecture.

### Final Ensemble Strategy

A stacking ensemble was constructed using:

- Extra Trees Classifier
- XGBoost
- Logistic Regression

The objective was to combine the strengths of tree-based bagging, gradient boosting, and linear classification approaches to achieve better generalization and overall predictive performance.

In [4]:
import pandas as pd, numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.model_selection import  RandomizedSearchCV
import warnings
warnings.filterwarnings('ignore')


In [5]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')   

In [6]:
x_train = train_df.drop(columns=['Activity'])
y_train = train_df['Activity']
x_test = test_df.drop(columns=['Activity'])
y_test = test_df['Activity']

In [7]:
print('Calculating corealtion matrix')
corr_matrix =  x_train.corr().abs()

print('selecting upper triangle')
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape),k=1).astype(bool))

print('removing highly corelated columns ')
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.85)]

final_x_train = x_train.drop(columns=to_drop)

final_x_test = x_test.drop(columns=to_drop)

Calculating corealtion matrix
selecting upper triangle
removing highly corelated columns 


In [22]:
y_train.value_counts()

Activity
LAYING                1407
STANDING              1374
SITTING               1286
WALKING               1226
WALKING_UPSTAIRS      1073
WALKING_DOWNSTAIRS     986
Name: count, dtype: int64

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
    ('scaler', StandardScaler(), final_x_train.columns)
])

final_y_train = y_train.map({'LAYING':0, 'STANDING':1, 'SITTING':2, 'WALKING':3, 'WALKING_UPSTAIRS':4, 'WALKING_DOWNSTAIRS':5})
final_y_test = y_test.map({'LAYING':0, 'STANDING':1, 'SITTING':2, 'WALKING':3, 'WALKING_UPSTAIRS':4, 'WALKING_DOWNSTAIRS':5})


In [24]:
models = {
    'Random Forest': RandomForestClassifier(),
    'Extra Trees': ExtraTreesClassifier(),
}
parameters = {
    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [ 10, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'bootstrap': [True, False]
    },
    'Extra Trees': {
        'n_estimators': [100, 200, 300],
        'max_depth': [ 10, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'bootstrap': [True, False]
    }
}

In [27]:
best_models = {}
for model_name, model in models.items():
    print(f"Training {model_name}...")
    my_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    current_parameters = {}
    for param_name, param_values in parameters[model_name].items():
        current_parameters[f'classifier__{param_name}'] = param_values
    random_search = RandomizedSearchCV(my_pipeline,
                                        current_parameters,
                                        n_iter=10,
                                        cv=5, 
                                        scoring='f1_macro',
                                        n_jobs=-1,
                                        random_state=42)
    random_search.fit(final_x_train, final_y_train)
    best_models[model_name] = random_search.best_estimator_
    print(f"Optimal Parameters for {model_name}: {random_search.best_params_}")
    print(f"Best F1 Score for {model_name}: {random_search.best_score_}")

Training Random Forest...
Optimal Parameters for Random Forest: {'classifier__n_estimators': 100, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 4, 'classifier__max_depth': 20, 'classifier__bootstrap': True}
Best F1 Score for Random Forest: 0.9216626152168315
Training Extra Trees...
Optimal Parameters for Extra Trees: {'classifier__n_estimators': 300, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 1, 'classifier__max_depth': 20, 'classifier__bootstrap': False}
Best F1 Score for Extra Trees: 0.9250915800204821


In [28]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report



print("\n--- MODELS EVALUATION STARTED ---")
for name, model in best_models.items():
    print(f"\nEvaluating: {name}...")
    
    y_pred = model.predict(final_x_test)
    
    accuracy = accuracy_score(final_y_test, y_pred)
    f1 = f1_score(final_y_test, y_pred, average='macro')
    classification_rep = classification_report(final_y_test, y_pred)

    print(f"Accuracy: {accuracy}")
    print(f"F1 Score: {f1}")
    print("Classification Report:\n", classification_rep)
    print('done')


--- MODELS EVALUATION STARTED ---

Evaluating: Random Forest...
Accuracy: 0.9317950458092976
F1 Score: 0.9309618884990436
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       537
           1       0.89      0.89      0.89       532
           2       0.89      0.89      0.89       491
           3       0.93      0.99      0.96       496
           4       0.91      0.91      0.91       471
           5       0.97      0.90      0.93       420

    accuracy                           0.93      2947
   macro avg       0.93      0.93      0.93      2947
weighted avg       0.93      0.93      0.93      2947

done

Evaluating: Extra Trees...
Accuracy: 0.9429928741092637
F1 Score: 0.9420098691151022
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       537
           1       0.89      0.95      0.92       532
           2       0.94      0.8

In [30]:
import joblib
model = best_models['Extra Trees']
joblib.dump(model, 'ExtraTrees_model.pkl')

['ExtraTrees_model.pkl']

## NOW FINAL STACKING

In [2]:
import joblib
from sklearn.ensemble import StackingClassifier
extra_trees_model = joblib.load('ExtraTrees_model.pkl')
svm_model = joblib.load('svm_model.pkl')
xgb_model = joblib.load('Xgb_pipe.pkl')



c:\Program Files\Python314\Lib\pickle.py:1835: UserWarning: [16:07:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\gbm\../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


In [11]:
base_models = [
    ('extra_trees', extra_trees_model),
    ('svm', svm_model),
    ('xgb', xgb_model)
]

stacked_model = StackingClassifier(estimators=base_models,
                                    final_estimator=LogisticRegression(max_iter=1500),
                                    cv='prefit'
                                    )
stacked_model.fit(final_x_train, final_y_train)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.The type of estimator is generally expected to be a classifier.However, one can pass a regressor for some use case (e.g. ordinalregression).","[('extra_trees', ...), ('svm', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA classifier which will be used to combine the base estimators.The default classifier is a:class:`~sklearn.linear_model.LogisticRegression`.",LogisticRegre...max_iter=1500)
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",'prefit'
,"stack_method stack_method: {'auto', 'predict_proba', 'decision_function', 'predict'}, default='auto'Methods called for each base estimator. It can be:* if 'auto', it will try to invoke, for each estimator, `'predict_proba'`, `'decision_function'` or `'predict'` in that order.* otherwise, one of `'predict_proba'`, `'decision_function'` or `'predict'`. If the method is not implemented by the estimator, it will raise an error.",'auto'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary <n_jobs>` for more details.",None
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,) or list of ndarray if `y` is of type `""multilabel-indicator""`.Class labels.","ndarray[int64](6,)","[0,1,2,3,4,5]"
"estimators_ estimators_: list of estimatorsThe elements of the `estimators` parameter, having been fitted on thetraining data. If an estimator has been set to `'drop'`, itwill not appear in `estimators_`. When `cv=""prefit""`, `estimators_`is set to `estimators` and is not fitted again.",list,"[Pipeline(step...mators=300))]), Pipeline(step...l='linear'))]), Pipeline(step...=None, ...))])]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimators expose such an attribute

In [12]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
predictions = stacked_model.predict(final_x_test)
print("Stacked Model Evaluation:")
accuracy = accuracy_score(final_y_test, predictions)
f1 = f1_score(final_y_test, predictions, average='macro')
print(f"Accuracy: {accuracy}")
print(f"F1 Score: {f1}")


Stacked Model Evaluation:
Accuracy: 0.9416355615880556
F1 Score: 0.9413220878525714


In [13]:
import joblib
joblib.dump(stacked_model, 'stacked_model.pkl')

['stacked_model.pkl']